In [ ]:
import json
import random
from google.colab import drive

drive.mount('/content/drive')

with open('/content/drive/MyDrive/flan_t5_results.json') as f:
    flan = sorted(json.load(f), key=lambda x: x['idx'])
with open('/content/drive/MyDrive/llama_results.json') as f:
    llama = sorted(json.load(f), key=lambda x: x['idx'])
with open('/content/drive/MyDrive/bias_labels.json') as f:
    bias_labels = json.load(f)

trap_idx = set(b['idx'] for b in bias_labels)

llama_errors = set(r['idx'] for r in llama if r['correct'] == 0)

non_trap_errors = sorted(llama_errors - trap_idx)

print(f"Trap questions (Llama wrong, Flan right): {len(trap_idx)}")
print(f"All Llama errors: {len(llama_errors)}")
print(f"Non-trap Llama errors (Llama wrong, Flan also wrong): {len(non_trap_errors)}")

random.seed(42)
sample_idx = sorted(random.sample(non_trap_errors, 150))
print(f"\nSampled 150 non-trap Llama errors for bias classification")
print(f"First 10 sampled indices: {sample_idx[:10]}")

with open('/content/drive/MyDrive/non_trap_sample_idx.json', 'w') as f:
    json.dump(sample_idx, f)
print(f"Saved to /content/drive/MyDrive/non_trap_sample_idx.json")

In [ ]:
import requests
import json
import time
from datasets import load_dataset
from google.colab import userdata

TOGETHER_API_KEY = userdata.get('TOGETHER_API_KEY')

dataset = load_dataset("GBaker/MedQA-USMLE-4-options", trust_remote_code=True)
with open('/content/drive/MyDrive/llama_results.json') as f:
    llama = {r['idx']: r for r in json.load(f)}
with open('/content/drive/MyDrive/non_trap_sample_idx.json') as f:
    sample_idx = json.load(f)

print("Dataset schema (first row keys):", list(dataset['test'][0].keys()))
print("First row sample:", {k: str(v)[:100] for k, v in dataset['test'][0].items()})

In [ ]:
def classify_bias(question, options, gold, wrong_answer, api_key):
    gold_text = options.get(gold, '')
    wrong_text = options.get(wrong_answer, '')
    opts_text = "\n".join([f"{k}: {v}" for k, v in options.items()])

    prompt = f"""You are an expert medical educator. Classify the cognitive bias that caused this AI model error.

Question: {question}

Options:
{opts_text}

Correct answer: {gold} - {gold_text}
Wrong answer chosen: {wrong_answer} - {wrong_text}

Reply with ONLY one of these labels:
- AVAILABILITY_BIAS (over-weighting salient/memorable symptoms)
- ANCHORING_BIAS (over-relying on first piece of information)
- FRAMING_EFFECT (different conclusion from same info presented differently)
- PREMATURE_CLOSURE (stopping reasoning too early)
- OTHER

Single label only, no explanation:"""

    try:
        r = requests.post(
            "https://api.together.xyz/v1/chat/completions",
            headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
            json={
                "model": "meta-llama/Llama-3.3-70B-Instruct-Turbo",
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": 10,
                "temperature": 0.0
            },
            timeout=30
        )
        text = r.json()['choices'][0]['message']['content'].strip().upper()
        for label in ['AVAILABILITY_BIAS', 'ANCHORING_BIAS',
                      'FRAMING_EFFECT', 'PREMATURE_CLOSURE', 'OTHER']:
            if label in text:
                return label
        return 'OTHER'
    except Exception as e:
        print(f"  Error: {e}")
        return 'ERROR'

print(f"Classifying bias on {len(sample_idx)} non-trap Llama errors...")
print(f"Model: Llama-3.3-70B-Instruct-Turbo (same as original)")
print(f"Estimated cost: ~$0.35\n")

non_trap_bias_labels = []

for i, idx in enumerate(sample_idx):
    q = dataset['test'][idx]
    llama_pred = llama[idx]['pred']

    if llama_pred == q['answer_idx']:
        print(f"  WARNING: idx {idx} is not actually a Llama error, skipping")
        continue

    label = classify_bias(
        question=q['question'],
        options=q['options'],
        gold=q['answer_idx'],
        wrong_answer=llama_pred,
        api_key=TOGETHER_API_KEY
    )

    non_trap_bias_labels.append({'idx': idx, 'bias_type': label})

    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{len(sample_idx)} done...")

    time.sleep(0.3)

with open('/content/drive/MyDrive/non_trap_bias_labels.json', 'w') as f:
    json.dump(non_trap_bias_labels, f)

from collections import Counter
counts = Counter(b['bias_type'] for b in non_trap_bias_labels)
total = len(non_trap_bias_labels)

print(f"\n=== Non-trap Llama errors bias distribution (n={total}) ===")
for bias, count in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"  {bias}: {count} ({count/total*100:.1f}%)")

print(f"\nSaved to /content/drive/MyDrive/non_trap_bias_labels.json")

In [ ]:
import requests, json, time
from collections import Counter

TOGETHER_API_KEY = userdata.get('TOGETHER_API_KEY')
print(f"Key starts with: {TOGETHER_API_KEY[:12]}...")
print(f"Key length: {len(TOGETHER_API_KEY)}")

print("\n--- Test call ---")
r = requests.post(
    "https://api.together.xyz/v1/chat/completions",
    headers={"Authorization": f"Bearer {TOGETHER_API_KEY}", "Content-Type": "application/json"},
    json={
        "model": "meta-llama/Llama-3.3-70B-Instruct-Turbo",
        "messages": [{"role": "user", "content": "Reply with only the word PING"}],
        "max_tokens": 5,
        "temperature": 0.0
    },
    timeout=30
)
print(f"Status code: {r.status_code}")
print(f"Response: {r.text[:500]}")

if r.status_code != 200 or 'choices' not in r.json():
    print("\n>>> STOP. Fix auth before proceeding. <<<")
else:
    print("\n>>> Test passed. Running classification loop... <<<\n")

    non_trap_bias_labels = []
    error_count = 0

    for i, idx in enumerate(sample_idx):
        q = dataset['test'][idx]
        llama_pred = llama[idx]['pred']

        if llama_pred == q['answer_idx']:
            continue

        label = classify_bias(
            question=q['question'],
            options=q['options'],
            gold=q['answer_idx'],
            wrong_answer=llama_pred,
            api_key=TOGETHER_API_KEY
        )

        if label == 'ERROR':
            error_count += 1
            if error_count > 5:
                print(f"  >>> {error_count} errors in a row, aborting <<<")
                break
        else:
            error_count = 0

        non_trap_bias_labels.append({'idx': idx, 'bias_type': label})

        if (i + 1) % 25 == 0:
            print(f"  {i+1}/{len(sample_idx)} done...")

        time.sleep(0.3)

    with open('/content/drive/MyDrive/non_trap_bias_labels.json', 'w') as f:
        json.dump(non_trap_bias_labels, f)

    counts = Counter(b['bias_type'] for b in non_trap_bias_labels)
    total = len(non_trap_bias_labels)
    print(f"\n=== Non-trap Llama errors bias distribution (n={total}) ===")
    for bias, count in sorted(counts.items(), key=lambda x: -x[1]):
        print(f"  {bias}: {count} ({count/total*100:.1f}%)")

In [ ]:
import json, requests

test_idx = sample_idx[0]
q = dataset['test'][test_idx]
llama_pred = llama[test_idx]['pred']

print(f"Test idx: {test_idx}")
print(f"Gold: {q['answer_idx']}, Llama chose: {llama_pred}")
print(f"Question length: {len(q['question'])} chars")
print(f"Options: {list(q['options'].keys())}")

opts_text = "\n".join([f"{k}: {v}" for k, v in q['options'].items()])
prompt = f"""You are an expert medical educator. Classify the cognitive bias that caused this AI model error.

Question: {q['question']}

Options:
{opts_text}

Correct answer: {q['answer_idx']} - {q['options'].get(q['answer_idx'], '')}
Wrong answer chosen: {llama_pred} - {q['options'].get(llama_pred, '')}

Reply with ONLY one of these labels:
- AVAILABILITY_BIAS (over-weighting salient/memorable symptoms)
- ANCHORING_BIAS (over-relying on first piece of information)
- FRAMING_EFFECT (different conclusion from same info presented differently)
- PREMATURE_CLOSURE (stopping reasoning too early)
- OTHER

Single label only, no explanation:"""

print(f"\nPrompt length: {len(prompt)} chars")
print("\n--- Calling API ---")

r = requests.post(
    "https://api.together.xyz/v1/chat/completions",
    headers={"Authorization": f"Bearer {TOGETHER_API_KEY}", "Content-Type": "application/json"},
    json={
        "model": "meta-llama/Llama-3.3-70B-Instruct-Turbo",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 10,
        "temperature": 0.0
    },
    timeout=30
)

print(f"Status code: {r.status_code}")
print(f"Full response:\n{r.text[:1500]}")

In [ ]:
try:
    with open('/content/drive/MyDrive/non_trap_bias_labels.json') as f:
        current = json.load(f)
    print(f"Saved entries: {len(current)}")
    from collections import Counter
    counts = Counter(b['bias_type'] for b in current)
    for k, v in counts.most_common():
        print(f"  {k}: {v}")
except Exception as e:
    print(f"No saved file or error: {e}")

In [ ]:
import json, requests, time
from collections import Counter
import numpy as np
from scipy import stats

def classify_bias_v2(question, options, gold, wrong_answer, api_key):
    gold_text = options.get(gold, '')
    wrong_text = options.get(wrong_answer, '')
    opts_text = "\n".join([f"{k}: {v}" for k, v in options.items()])
    prompt = f"""You are an expert medical educator. Classify the cognitive bias that caused this AI model error.

Question: {question}

Options:
{opts_text}

Correct answer: {gold} - {gold_text}
Wrong answer chosen: {wrong_answer} - {wrong_text}

Reply with ONLY one of these labels:
- AVAILABILITY_BIAS (over-weighting salient/memorable symptoms)
- ANCHORING_BIAS (over-relying on first piece of information)
- FRAMING_EFFECT (different conclusion from same info presented differently)
- PREMATURE_CLOSURE (stopping reasoning too early)
- OTHER

Single label only, no explanation:"""

    r = requests.post(
        "https://api.together.xyz/v1/chat/completions",
        headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
        json={
            "model": "meta-llama/Llama-3.3-70B-Instruct-Turbo",
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 10,
            "temperature": 0.0
        },
        timeout=30
    )
    if r.status_code != 200:
        return 'ERROR', f"HTTP {r.status_code}: {r.text[:200]}"
    try:
        text = r.json()['choices'][0]['message']['content'].strip().upper()
    except (KeyError, IndexError):
        return 'ERROR', f"Bad shape: {r.text[:200]}"
    for label in ['AVAILABILITY_BIAS', 'ANCHORING_BIAS',
                  'FRAMING_EFFECT', 'PREMATURE_CLOSURE', 'OTHER']:
        if label in text:
            return label, None
    return 'OTHER', None

with open('/content/drive/MyDrive/non_trap_bias_labels.json') as f:
    non_trap = json.load(f)
with open('/content/drive/MyDrive/bias_labels.json') as f:
    trap = json.load(f)

error_entries = [b for b in non_trap if b['bias_type'] == 'ERROR']
print(f"Retrying {len(error_entries)} ERROR entries...")
for entry in error_entries:
    idx = entry['idx']
    q = dataset['test'][idx]
    llama_pred = llama[idx]['pred']
    label, err = classify_bias_v2(q['question'], q['options'], q['answer_idx'], llama_pred, TOGETHER_API_KEY)
    if label != 'ERROR':
        entry['bias_type'] = label
        print(f"  idx={idx}: {label}")
    else:
        print(f"  idx={idx}: still failing — {err}")
    time.sleep(0.3)

with open('/content/drive/MyDrive/non_trap_bias_labels.json', 'w') as f:
    json.dump(non_trap, f)

trap_counts = Counter(b['bias_type'] for b in trap)
non_trap_counts = Counter(b['bias_type'] for b in non_trap if b['bias_type'] != 'ERROR')

categories = ['PREMATURE_CLOSURE', 'ANCHORING_BIAS', 'AVAILABILITY_BIAS', 'OTHER']

print("\n=== Final comparison ===")
print(f"{'Category':<22} {'Trap':<18} {'Non-trap':<18}")
print("-" * 60)
for cat in categories:
    t = trap_counts.get(cat, 0)
    nt = non_trap_counts.get(cat, 0)
    t_pct = t / sum(trap_counts.values()) * 100
    nt_pct = nt / sum(non_trap_counts.values()) * 100
    print(f"{cat:<22} {t:>4} ({t_pct:>5.1f}%)        {nt:>4} ({nt_pct:>5.1f}%)")

trap_pooled = [
    trap_counts.get('PREMATURE_CLOSURE', 0),
    trap_counts.get('ANCHORING_BIAS', 0),
    trap_counts.get('AVAILABILITY_BIAS', 0) + trap_counts.get('OTHER', 0)
]
non_trap_pooled = [
    non_trap_counts.get('PREMATURE_CLOSURE', 0),
    non_trap_counts.get('ANCHORING_BIAS', 0),
    non_trap_counts.get('AVAILABILITY_BIAS', 0) + non_trap_counts.get('OTHER', 0)
]

contingency = np.array([trap_pooled, non_trap_pooled])
print(f"\nContingency table (pooled AV+OTHER):")
print(f"  Trap:     {trap_pooled}  (total {sum(trap_pooled)})")
print(f"  Non-trap: {non_trap_pooled}  (total {sum(non_trap_pooled)})")

chi2, p, dof, expected = stats.chi2_contingency(contingency)
print(f"\nChi-square test of independence:")
print(f"  chi2 = {chi2:.3f}, dof = {dof}, p = {p:.4f}")

std_residuals = (contingency - expected) / np.sqrt(expected)
print(f"\nStandardized residuals (|z|>2 = notable):")
labels = ['PC', 'ANCHOR', 'AV+OTHER']
for i, group in enumerate(['Trap    ', 'Non-trap']):
    row = std_residuals[i]
    print(f"  {group}: " + "  ".join([f"{l}={r:+.2f}" for l, r in zip(labels, row)]))

pc_2x2 = np.array([
    [trap_counts.get('PREMATURE_CLOSURE', 0),
     sum(trap_counts.values()) - trap_counts.get('PREMATURE_CLOSURE', 0)],
    [non_trap_counts.get('PREMATURE_CLOSURE', 0),
     sum(non_trap_counts.values()) - non_trap_counts.get('PREMATURE_CLOSURE', 0)]
])
print(f"\n2x2 (PC vs not-PC):")
print(f"  Trap:     {pc_2x2[0]}")
print(f"  Non-trap: {pc_2x2[1]}")
odds_ratio, p_fisher = stats.fisher_exact(pc_2x2)
print(f"  Odds ratio: {odds_ratio:.3f}")
print(f"  Fisher exact p: {p_fisher:.4f}")